In [1]:
!pip install transformers torch scikit-learn -q

In [2]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score

In [3]:
texts = [
    "The clock stopped ticking at midnight for no reason",
    "I found a coin inside my shoe this morning",
    "Blue birds rarely visit this side of the city",
    "My phone battery drains faster in cold weather",
    "There is a strange smell coming from the hallway",
    "The book ended without any conclusion",
    "Someone left the lights on all night",
    "The keyboard keys feel sticky today",
    "Rain started suddenly without any clouds",
    "I forgot where I parked my car again",
    "The soup tastes different than yesterday",
    "A dog kept barking outside for hours",
    "The screen flickers when brightness is low",
    "I heard a loud noise from upstairs",
    "The road was completely empty this evening",
    "My internet slowed down unexpectedly",
    "The chair makes a noise when I sit",
    "I saw a shadow move across the wall",
    "The package arrived earlier than expected",
    "The air feels unusually dry today"
]

labels = [0, 1] * 10  # alternating labels (total 20)
print("Number of samples:" , len(texts))

Number of samples: 20


In [4]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
encodings= tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=32,
    return_tensors="pt"
)
print(encodings.keys())
print("Input IDs shape:", encodings["input_ids"].shape)
print("Attention Mask shape:", encodings["attention_mask"].shape)

KeysView({'input_ids': tensor([[  101,  1996,  5119,  3030, 28561,  2012,  7090,  2005,  2053,  3114,
           102],
        [  101,  1045,  2179,  1037,  9226,  2503,  2026, 10818,  2023,  2851,
           102],
        [  101,  2630,  5055,  6524,  3942,  2023,  2217,  1997,  1996,  2103,
           102],
        [  101,  2026,  3042,  6046, 18916,  5514,  1999,  3147,  4633,   102,
             0],
        [  101,  2045,  2003,  1037,  4326,  5437,  2746,  2013,  1996,  6797,
           102],
        [  101,  1996,  2338,  3092,  2302,  2151,  7091,   102,     0,     0,
             0],
        [  101,  2619,  2187,  1996,  4597,  2006,  2035,  2305,   102,     0,
             0],
        [  101,  1996,  9019,  6309,  2514, 15875,  2651,   102,     0,     0,
             0],
        [  101,  4542,  2318,  3402,  2302,  2151,  8044,   102,     0,     0,
             0],
        [  101,  1045,  9471,  2073,  1045,  9083,  2026,  2482,  2153,   102,
             0],
        [  101,  

Create Custom Dataset Class

In [6]:
class SentimentDataset(Dataset):
  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = labels

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, index):
    item={key: val[index] for key, val in self.encodings.items()}
    item["labels"] = torch.tensor(self.labels[index])
    return item

Prepare DataLoader

In [7]:
dataset= SentimentDataset(encodings, labels)
loader= DataLoader(dataset, batch_size=8, shuffle=True)


Load BERT Classification Model

In [8]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


cpu


Define Optimizer

In [9]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

Train the Model

In [10]:
model.train()

epochs = 3

for epoch in range(epochs):

    total_loss = 0

    for batch in loader:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        total_loss += loss.item()

        loss.backward()

        optimizer.step()

    avg_loss = total_loss / len(loader)

    print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

Epoch 1 Loss: 0.7099
Epoch 2 Loss: 0.6602
Epoch 3 Loss: 0.5948


Evaluate Model

In [11]:
model.eval()

predictions = []
true_labels = []

with torch.no_grad():

    for batch in loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

Accuracy Score

In [12]:
accuracy = accuracy_score(true_labels, predictions)

print("Accuracy:", accuracy)

Accuracy: 0.95


Test Custom Sentence

In [13]:
test_text = "I really enjoyed this movie"

encoding = tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=32
)

input_ids = encoding["input_ids"].to(device)
attention_mask = encoding["attention_mask"].to(device)

model.eval()

with torch.no_grad():

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

prediction = torch.argmax(outputs.logits, dim=1)

print("Predicted Label:", prediction.item())

Predicted Label: 0
